# climate-toolkit — Rwanda maize & potato trial (control vs treatment)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/climate_toolkit_rwanda_trial_colab.ipynb)

A worked, project-manager-style walkthrough: use **climate-toolkit** to characterise the climate at **50 agricultural sites across Rwanda's five provinces** (maize and potato), then test whether a **treatment** (e.g. a drought-tolerant variety / climate-smart package) **outperforms the control** — and specifically whether its advantage *grows under climate stress* (drought, heat), which is what "climate resilience" means.

### What is real vs. simulated

| Layer | Source | Real? |
|---|---|---|
| Daily climate per site (rainfall, temperature) | climate-toolkit → NASA POWER / AgERA5 | **Real** observations |
| Per-site climate-stress index (water deficit, dry spells, heat days) | derived here from the real climate | **Real**, transparent formula |
| Site layout, crop assignment, control/treatment plots | designed here | Simulated trial design |
| Yields (control & treatment) | a documented synthetic model driven by the *real* stress index | **Simulated** — illustrative only |

> ⚠️ The toolkit fetches and analyses **real** climate; it does **not** model crop yields or run agronomic trials. The yield layer here is a clearly-labelled synthetic model so we can demonstrate the *analysis workflow*. Swap in your own measured yields to make it a real study.

### Contents
1. Install
2. Configure the data source (NASA POWER by default; Earth Engine optional)
3. Design the trial — 50 sites × 5 provinces, maize & potato, control + treatment
4. Project-manager tour of the toolkit on **one** representative site (all seven functions)
5. Batch: extract a real climate-stress index for all 50 sites
6. Synthetic yield model — control vs treatment, driven by real stress
7. Does the treatment win? Does it confer *climate resilience*?
8. Where to go next

## 1. Install

Not on PyPI yet — install from GitHub (~1–2 min on Colab).

In [ ]:
%pip install -q "git+https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit.git"

In [ ]:
import climate_toolkit as ct

print(f"climate_toolkit v{ct.__version__}")
print("Public API:", [n for n in ct.__all__ if not n.startswith("__")])

## 2. Configure the data source

Most of the toolkit's gridded sources run through **Google Earth Engine** (a free, one-time setup — see the [main quick-start notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/climate_toolkit_colab.ipynb)). This trial notebook defaults to **`nasa_power`**, which needs **no credentials** and runs anywhere — so all 50 sites work out of the box.

If you have completed the Earth Engine setup and want the richer AgERA5 grid instead, set `RUN_EARTH_ENGINE = True` and your project id below; the whole batch switches to `agera_5`.

In [ ]:
RUN_EARTH_ENGINE = False          # True to use AgERA5 (needs a registered GEE project)
GCP_PROJECT_ID = "your-ee-project-id"

if RUN_EARTH_ENGINE:
    import os
    import ee

    ee.Authenticate()
    os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
    ee.Initialize(project=GCP_PROJECT_ID)
    SOURCE = "agera_5"
    print("Earth Engine ready — using AgERA5.")
else:
    SOURCE = "nasa_power"
    print("Using NASA POWER (credential-free).")

## 3. Design the trial

**Rwanda has five provinces.** We place **10 sites in each** (50 total), jittered around agricultural centroids, and assign a crop by the province's typical agro-ecology:

- **Northern & Western** (high-altitude volcanic / Congo-Nile ridge) → **potato**
- **Eastern & Southern & Kigali** (warmer, drier lowlands & central plateau) → **maize**

Every site carries two plots — a **control** and a **treatment** — so the yield comparison below is *paired* within site-year. The layout is deterministic (fixed seed) so the notebook reproduces exactly.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Province centroids in agricultural zones (lat, lon) and their default crop
PROVINCES = {
    "Kigali":   {"lat": -1.94, "lon": 30.06, "crop": "maize"},
    "Northern": {"lat": -1.50, "lon": 29.61, "crop": "potato"},
    "Southern": {"lat": -2.48, "lon": 29.74, "crop": "maize"},
    "Eastern":  {"lat": -1.78, "lon": 30.44, "crop": "maize"},
    "Western":  {"lat": -2.05, "lon": 29.35, "crop": "potato"},
}
SITES_PER_PROVINCE = 10

rows = []
for prov, info in PROVINCES.items():
    for i in range(SITES_PER_PROVINCE):
        rows.append({
            "site_id": f"{prov[:3].upper()}-{i+1:02d}",
            "province": prov,
            "crop": info["crop"],
            # ~0.15 deg jitter (~15 km) around the centroid, still on land
            "lat": round(info["lat"] + rng.uniform(-0.15, 0.15), 4),
            "lon": round(info["lon"] + rng.uniform(-0.15, 0.15), 4),
        })

sites = pd.DataFrame(rows)
print(f"{len(sites)} sites, {sites.crop.value_counts().to_dict()}")
sites.head()

In [ ]:
# Quick look at where the sites sit, coloured by crop
ax = sites.plot.scatter(
    x="lon", y="lat",
    c=sites["crop"].map({"maize": "tab:orange", "potato": "tab:brown"}),
    figsize=(6, 6), s=40, title="Simulated trial sites across Rwanda (orange=maize, brown=potato)",
)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude");

## 4. Project-manager tour of the toolkit (one site)

Before batching all 50 sites, walk the **whole toolkit** on a single representative site — the way a project manager would scope a location. We use one Eastern-province maize site (the drier zone, where drought stress matters most).

We cover all seven public functions:

1. `fetch_climate_data` — daily climate
2. `analyze_climate_statistics` — seasonal climatology & water balance
3. `evaluate_hazards` — crop-specific drought/heat hazard indicators
4. `compare_climate_periods` — a focal year vs. the long-term baseline
5. `compare_climate_sources` — cross-check two datasets *(needs Earth Engine)*
6. `download_station_data` — nearest weather-station observations
7. `compare_station_to_grids` — validate the grid against the station

In [ ]:
from datetime import date
from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

tour = sites[sites.province == "Eastern"].iloc[0]
TOUR_COORD = (float(tour.lat), float(tour.lon))
print(f"Tour site: {tour.site_id}  {TOUR_COORD}  crop={tour.crop}")

VARS = [
    ClimateVariable.precipitation,
    ClimateVariable.max_temperature,
    ClimateVariable.min_temperature,
]

### 4.1 `fetch_climate_data` — daily climate

In [ ]:
df_tour = ct.fetch_climate_data(
    source=SOURCE,
    location_coord=TOUR_COORD,
    variables=VARS,
    date_from=date(2018, 1, 1),
    date_to=date(2023, 12, 31),
    verbose=False,
)
print("Shape:", df_tour.shape, "| columns:", df_tour.columns.tolist())
df_tour.head()

In [ ]:
df_tour.set_index("date")["precipitation"].plot(
    figsize=(11, 3), title=f"Daily precipitation 2018–2023 — {tour.site_id}"
);

### 4.2 `analyze_climate_statistics` — seasonal climatology & water balance

We pass a **fixed season** (`"MM-DD:MM-DD"`) for Rwanda's long-rains season B (Feb–Jun) so the long-term means compare like with like across years.

In [ ]:
stats_tour = ct.analyze_climate_statistics(
    location_coord=TOUR_COORD,
    start_year=2018, end_year=2023,
    source=SOURCE,
    fixed_season="02-01:06-30",
)
ltm = stats_tour["ltm_season_summary"]
print("LTM mode:", ltm["mode"])
pd.json_normalize(ltm["windows"])

### 4.3 `evaluate_hazards` — drought & heat indicators

This is the bridge to our trial: the hazard block reports **water-stress days (NDWS)**, **dry days (NDD)**, and **heat days (NTx35/NTx40)** — the very stresses a drought-tolerant treatment is meant to buffer.

In [ ]:
hz = ct.evaluate_hazards(
    crop_name=tour.crop,
    location_coord=TOUR_COORD,
    date_from="2023-02-01", date_to="2023-06-30",
    source=SOURCE,
)

# evaluate_hazards nests hazard indicators under "assessments" (one entry per
# detected season) or, for a single season, at the top level. Handle both.
assessments = hz.get("assessments") or [hz]

def hazard_table(assessment):
    he = assessment["hazard_evaluation"]
    return pd.DataFrame([
        {"indicator": k,
         "value": v.get("value_days", v.get("value_thi", v.get("value_humidex"))),
         "status": v.get("status")}
        for k, v in he.items()
    ])

for i, a in enumerate(assessments, 1):
    info = a.get("season_info", {})
    print(f"Season {i}: {info.get('onset_date', '?')} -> {info.get('cessation_date', '?')}")
    display(hazard_table(a))

### 4.4 `compare_climate_periods` — focal year vs. baseline

How did 2023 compare against the 2018–2022 local baseline? (A full 1991–2020 baseline is more robust but slower; we keep it short for the tour.)

In [ ]:
periods_tour = ct.compare_climate_periods(
    location=TOUR_COORD,
    baseline_start=2018, baseline_end=2022,
    focal_year=2023,
    source=SOURCE,
    fixed_season="02-01:06-30",
)
print("Comparison blocks:", sorted(periods_tour.keys()))

### 4.5 `compare_climate_sources` — cross-check datasets *(Earth Engine)*

Comparing gridded products needs at least one Earth Engine source, so this cell runs only when `RUN_EARTH_ENGINE = True`.

In [ ]:
if RUN_EARTH_ENGINE:
    import os
    cmp = ct.compare_climate_sources(
        sources=["nasa_power", "agera_5"],
        lat=TOUR_COORD[0], lon=TOUR_COORD[1],
        start="2023-01-01", end="2023-12-31",
        output_dir="./outputs",
    )
    print("Wrote:", sorted(os.listdir("./outputs")))
else:
    print("Skipped — set RUN_EARTH_ENGINE = True (section 2) to compare NASA POWER vs AgERA5.")

### 4.6 `download_station_data` — nearest station observations

In [ ]:
try:
    st = ct.download_station_data(
        station_source="ghcn_daily",
        station_coord=TOUR_COORD,
        date_from=date(2020, 1, 1), date_to=date(2020, 12, 31),
        max_distance_km=80.0, auto_select="auto-1",
    )
    print("Station rows:", len(st))
    display(st.head())
except Exception as e:
    print("No usable station near this site:", type(e).__name__, str(e)[:120])

### 4.7 `compare_station_to_grids` — validate the grid

How well does the grid match ground observations at this site? (Precipitation only — nearby stations are often too sparse on temperature to pass the completeness guard.)

In [ ]:
try:
    val = ct.compare_station_to_grids(
        station_source="ghcn_daily",
        station_coord=TOUR_COORD,
        date_from=date(2019, 1, 1), date_to=date(2020, 12, 31),
        grid_sources=[SOURCE],
        variables=[ClimateVariable.precipitation],
        max_distance_km=80.0,
        verbose=False,
    )
    print("Validation blocks:", sorted(val.keys()))
    display(val["confidence_summary"])
except Exception as e:
    print("Validation unavailable for this site:", type(e).__name__, str(e)[:120])

## 5. Batch: a real climate-stress index for all 50 sites

Now the project scales out. For each site we fetch the real daily climate once (2018–2023) and derive a **transparent, per-season-year stress index** from Rwanda's long-rains season (Feb–Jun) — the growing window for both crops.

The index combines three real, interpretable drivers, each normalised to 0–1 and clipped. In Rwanda's generous rains, *seasonal totals* are rarely the problem — **badly distributed rainfall (long dry spells within the season)** is the real drought mechanism, so it carries the most weight:

- **Longest dry spell** — consecutive rain-free days in the season (the dominant driver here)
- **Water deficit** — how far seasonal rainfall falls short of the crop's water need
- **Heat days** — days above the crop's high-temperature threshold

$$\text{stress} = \operatorname{clip}\big(0.45\,\text{deficit} + 0.40\,\tfrac{\text{dry spell}}{30} + 0.15\,\tfrac{\text{heat days}}{30},\;0,\;1\big)$$

Higher stress = a harder season. We analyse **2015–2023**, which spans the 2016–2017 East African drought, so the index varies across a meaningful range. This is the real climate signal that drives the (simulated) yields in section 6.

In [ ]:
# Crop agronomic constants for the stress index
CROP = {
    "maize":  {"season_water_need_mm": 450.0, "heat_threshold_c": 30.0},
    "potato": {"season_water_need_mm": 500.0, "heat_threshold_c": 28.0},
}
SEASON_MONTHS = [2, 3, 4, 5, 6]   # Feb–Jun long rains
YEARS = range(2015, 2024)


def longest_dry_spell(precip, wet_mm=1.0):
    """Longest run of days with precip < wet_mm."""
    longest = run = 0
    for p in precip:
        if p < wet_mm:
            run += 1
            longest = max(longest, run)
        else:
            run = 0
    return longest


def season_stress(df, crop):
    """Per-year stress index (0–1) for the Feb–Jun season, from real daily data."""
    c = CROP[crop]
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d[d["date"].dt.month.isin(SEASON_MONTHS)]
    out = []
    for year, g in d.groupby(d["date"].dt.year):
        rain = g["precipitation"].fillna(0).to_numpy()
        tmax = g["max_temperature"].to_numpy()
        deficit = max(0.0, (c["season_water_need_mm"] - rain.sum()) / c["season_water_need_mm"])
        dry = longest_dry_spell(rain)
        heat = int((tmax > c["heat_threshold_c"]).sum())
        stress = np.clip(0.45 * deficit + 0.40 * (dry / 30) + 0.15 * (heat / 30), 0, 1)
        out.append({"year": int(year), "season_rain_mm": round(float(rain.sum()), 1),
                    "dry_spell_days": int(dry), "heat_days": int(heat),
                    "stress": round(float(stress), 3)})
    return pd.DataFrame(out)

In [ ]:
# Fetch each site once and compute its per-year stress. ~50 fetches; cached, so re-runs are fast.
records = []
for n, site in enumerate(sites.itertuples(), 1):
    try:
        df_site = ct.fetch_climate_data(
            source=SOURCE,
            location_coord=(site.lat, site.lon),
            variables=VARS,
            date_from=date(2015, 1, 1), date_to=date(2023, 12, 31),
            verbose=False,
        )
        s = season_stress(df_site, site.crop)
        s["site_id"] = site.site_id
        s["province"] = site.province
        s["crop"] = site.crop
        records.append(s)
    except Exception as e:
        print(f"  {site.site_id} failed: {type(e).__name__} {str(e)[:80]}")
    if n % 10 == 0:
        print(f"  fetched {n}/{len(sites)} sites")

stress = pd.concat(records, ignore_index=True)
print("Site-years:", len(stress))
stress.groupby("province")["stress"].describe()[["mean", "min", "max"]].round(3)

## 6. Synthetic yield model — control vs treatment

The toolkit stops at climate; the yields below are a **documented synthetic model** so we can demonstrate the analysis. Real climate stress from section 5 is the driver.

For each site-year and plot:

```
control   = base × (1 − sensitivity × stress) × (1 + noise)
treatment = base × (1 + agronomic_gain) × (1 − sensitivity × (1 − tolerance) × stress) × (1 + noise)
```

- **`base`** — attainable yield with no stress (maize 4.0 t/ha, potato 20.0 t/ha)
- **`sensitivity`** — fraction of yield lost at maximum stress
- **`tolerance`** — the treatment's drought buffering: it removes this fraction of the stress penalty. **This is the climate-resilience knob.**
- **`agronomic_gain`** — a small baseline uplift the treatment gives even with no stress

Because `tolerance` shrinks the *stress-dependent* loss, the treatment's advantage should **grow as stress rises** — the signature of climate resilience, which we test in section 7.

In [ ]:
YIELD = {
    "maize":  {"base": 4.0,  "sensitivity": 0.60},
    "potato": {"base": 20.0, "sensitivity": 0.50},
}
TOLERANCE = 0.55        # treatment removes 55% of the stress penalty (resilience)
AGRONOMIC_GAIN = 0.03   # +3% baseline uplift even without stress
NOISE_SD = 0.08         # plot-to-plot variability

ygen = np.random.default_rng(7)

def simulate(row):
    p = YIELD[row.crop]
    base, sens, stress = p["base"], p["sensitivity"], row.stress
    ctrl = base * (1 - sens * stress) * (1 + ygen.normal(0, NOISE_SD))
    trt = (base * (1 + AGRONOMIC_GAIN)
           * (1 - sens * (1 - TOLERANCE) * stress)
           * (1 + ygen.normal(0, NOISE_SD)))
    return pd.Series({"yield_control": round(ctrl, 3), "yield_treatment": round(trt, 3)})

trial = pd.concat([stress, stress.apply(simulate, axis=1)], axis=1)
trial["gap"] = (trial["yield_treatment"] - trial["yield_control"]).round(3)
# Relative gap (% of control) is scale-free, so maize and potato combine cleanly
trial["gap_pct"] = (100 * (trial["yield_treatment"] / trial["yield_control"] - 1)).round(2)
trial["stress_tercile"] = pd.qcut(trial["stress"], 3, labels=["low", "medium", "high"])
trial.head()

## 7. Does the treatment win — and is it *climate-resilient*?

Two distinct questions:

1. **Does treatment beat control overall?** → paired t-test across all site-years, per crop.
2. **Is the advantage climate-resilience, not just a flat bump?** → the treatment's **relative advantage (% over control) should increase with stress**. We use the *relative* gap so maize and potato (very different t/ha scales) combine cleanly. A positive, significant slope is the resilience signal; we also show it by stress tercile.

In [ ]:
from scipy import stats

# 1. Overall paired test (treatment vs control), per crop
for crop in ["maize", "potato"]:
    sub = trial[trial.crop == crop]
    t, p = stats.ttest_rel(sub["yield_treatment"], sub["yield_control"])
    lift = 100 * (sub["yield_treatment"].mean() / sub["yield_control"].mean() - 1)
    print(f"{crop:7s}: mean control={sub.yield_control.mean():.2f}, "
          f"treatment={sub.yield_treatment.mean():.2f} t/ha "
          f"(+{lift:.1f}%), paired t={t:.1f}, p={p:.2e}")

In [ ]:
# 2a. Resilience test: does the RELATIVE advantage grow with stress?
lr = stats.linregress(trial["stress"], trial["gap_pct"])
print(f"gap_pct = {lr.intercept:.1f} + {lr.slope:.1f} * stress")
print(f"slope p-value = {lr.pvalue:.2e}, R^2 = {lr.rvalue**2:.2f}")
print("Positive, significant slope => treatment advantage grows under stress (climate resilience).")

In [ ]:
# 2b. Advantage by stress tercile — the clearest view of resilience
tbl = (trial.groupby("stress_tercile", observed=True)
       .agg(mean_stress=("stress", "mean"),
            control=("yield_control", "mean"),
            treatment=("yield_treatment", "mean"),
            advantage_pct=("gap_pct", "mean"))
       .round(2))
print(tbl)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: yield vs stress, control vs treatment (maize, for one crop's scale)
m = trial[trial.crop == "maize"]
axes[0].scatter(m["stress"], m["yield_control"], s=18, alpha=0.6, label="control")
axes[0].scatter(m["stress"], m["yield_treatment"], s=18, alpha=0.6, label="treatment")
axes[0].set(xlabel="climate stress", ylabel="maize yield (t/ha)",
            title="Maize: yield vs. climate stress")
axes[0].legend()

# Right: relative advantage (%) vs stress with fit line (all site-years, both crops)
axes[1].scatter(trial["stress"], trial["gap_pct"], s=14, alpha=0.4, color="tab:green")
xs = np.linspace(trial["stress"].min(), trial["stress"].max(), 50)
axes[1].plot(xs, lr.intercept + lr.slope * xs, color="black", lw=2)
axes[1].axhline(0, color="grey", lw=0.8, ls="--")
axes[1].set(xlabel="climate stress", ylabel="treatment advantage (% over control)",
            title="Resilience: advantage grows with stress")
plt.tight_layout();

In [ ]:
# Province-level summary: where does the treatment help most?
prov = (trial.groupby("province")
        .agg(mean_stress=("stress", "mean"),
             control=("yield_control", "mean"),
             treatment=("yield_treatment", "mean"),
             gap=("gap", "mean"))
        .round(2)
        .sort_values("mean_stress", ascending=False))
print(prov)

## 8. Where to go next

What this notebook showed:

- climate-toolkit fetched **real** daily climate for 50 Rwandan sites and produced real seasonal, hazard, and validation outputs (section 4) and a real per-site climate-stress index (section 5).
- A transparent synthetic yield model (section 6) let us test a treatment. With `TOLERANCE = 0.55`, the treatment beat the control overall **and** its advantage grew with climate stress — the resilience signature (section 7).

To turn this into a real study:

- **Replace the synthetic yields** in section 6 with your measured control/treatment yields, keyed by `site_id` and `year`. Everything in section 7 then runs on real data.
- **Tune the stress index** (section 5) to your crop calendar and thresholds, or swap in `evaluate_hazards` indicators (NDWS, NDD) directly.
- **Upgrade the climate** to AgERA5 or CHIRPS by completing the Earth Engine setup and setting `RUN_EARTH_ENGINE = True`.
- Set `TOLERANCE = 0` to confirm the resilience signal disappears when the treatment has no drought buffering — a good sanity check on the analysis.

Docs: [Use as a package](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/use-as-a-package/) · [API reference](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/api/) · Issues: https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit/issues